# Final Project
## Data Loading and Preparation

In [12]:
import pandas as pd
import numpy as np

### Stock Prices and Daily Movements

In [13]:
# Load raw CSV (Capital IQ format with metadata header rows)
raw = pd.read_csv("CAPIQ Stocks and Daily Movement.csv", header=None)

# Column 0 is empty (leading comma); labels are in column 1, data starts at column 2
tickers = raw.iloc[1, 2:].values  # Row 1 = Ticker row

# Rows 0-7 are metadata; row 8 onward is daily price data
date_start = 8

# Build the prices DataFrame
prices = raw.iloc[date_start:, 2:].copy()
prices.columns = tickers
prices.index = pd.to_datetime(raw.iloc[date_start:, 1], format="%m/%d/%Y")
prices.index.name = "Date"

# Remove commas from numbers and convert to float
prices = prices.replace(",", "", regex=True).apply(pd.to_numeric, errors="coerce")

print(f"Shape: {prices.shape[0]} trading days x {prices.shape[1]} stocks")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Null count: {prices.isnull().sum().sum()}")
prices.head()

Shape: 1304 trading days x 113 stocks
Date range: 2021-01-01 to 2025-12-31
Null count: 0


,LLY,V,MA,CRM,IBM,LIN,PEP,OR,ACN,VZ,...,TKR,ITRI,LEA,MOTV3,NATU3,BIMBOA,GT,PINC,SIG,CRI
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-01,159.96,210.51,346.45,219.86,97.20,245.93,127.11,288.85,240.16,42.80,...,70.78,95.90,141.37,11.86,50.10,159.96,10.91,5.39,25.66,77.88
2021-01-04,156.79,209.57,341.16,217.67,95.70,241.54,123.65,289.69,235.79,42.88,...,69.12,95.65,139.95,11.68,49.62,156.79,10.17,5.49,26.02,76.33
2021-01-05,157.57,206.45,337.21,218.86,97.40,245.33,124.02,286.72,237.13,42.68,...,71.20,96.42,142.38,11.49,49.33,157.57,10.64,5.46,27.87,76.86
2021-01-06,155.68,204.63,337.34,213.56,99.83,253.38,122.51,287.00,239.73,43.20,...,76.18,106.98,147.29,11.10,46.67,155.68,10.78,5.50,31.11,83.24
2021-01-07,157.11,205.77,340.01,215.36,99.60,252.66,122.11,285.88,241.99,43.10,...,77.66,107.27,151.48,11.12,46.27,157.11,10.74,5.49,33.10,82.96


### WMEC Stocks and Info

In [14]:
# Load WMEC Stocks and Info
stocks_info = pd.read_csv("WMEC Stocks and Info.csv")

print(f"Shape: {stocks_info.shape[0]} companies x {stocks_info.shape[1]} columns")
print(f"Columns: {list(stocks_info.columns)}")
stocks_info.head()

Shape: 118 companies x 12 columns
Columns: ['Company', 'Symbol', 'Exchange', 'Structure', 'Sectors', 'Industry Groups', 'Industries', 'Sub-Industries', 'Domicile Country', 'Workforce Size', 'Annual Revenue', 'Workforce Representation']


,Company,Symbol,Exchange,Structure,Sectors,Industry Groups,Industries,Sub-Industries,Domicile Country,Workforce Size,Annual Revenue,Workforce Representation
0,CME Group,CME,NASDAQ,Public,Financials,Financial Services,Capital Markets,Financial Exchanges & Data,United States,"1,000 - 4,999 employees","$5,000,000,001 - $10,000,000,000",NaN
1,DocGo,DCGO,NASDAQ,Public,Health Care,Health Care Equipment & Services,Health Care Providers & Services,Health Care Services,United States,"1,000 - 4,999 employees","Less than $500,000,000",NaN
2,"Premier, Inc.",PINC,NASDAQ,Public,Health Care,Health Care Equipment & Services,Health Care Technology,Health Care Technology,United States,"1,000 - 4,999 employees","$1,000,000,001 - $5,000,000,000",NaN
3,TrueBlue Inc.,TBI,NYSE,Public,Consumer Discretionary,Consumer Services,Diversified Consumer Services,Specialized Consumer Services,United States,"1,000 - 4,999 employees","$1,000,000,001 - $5,000,000,000",NaN
4,A.O. Smith,AOS,NYSE,Public,Industrials,Capital Goods,Building Products,Building Products,United States,"10,000 - 14,999 employees","$1,000,000,001 - $5,000,000,000",31-40 percent


### Ethisphere / Solactive Backtest Data

In [15]:
# Load Performance sheet
ethic_perf = pd.read_excel(
    "Solactive_Ethic_Backtest_20260130.xlsx",
    sheet_name="Performance",
    usecols="A:E"  # Date + 4 index columns; skip the extra summary columns
)
ethic_perf["Date"] = pd.to_datetime(ethic_perf["Date"])
ethic_perf = ethic_perf.set_index("Date")

print(f"Performance shape: {ethic_perf.shape}")
print(f"Date range: {ethic_perf.index.min().date()} to {ethic_perf.index.max().date()}")
ethic_perf.head()

Performance shape: (1304, 4)
Date range: 2021-01-01 to 2025-12-31


,Ethics Premium PR,Ethics Premium GTR,SGMACUP Index,SGMACUT Index
Date,,,,
2021-01-01,1000.000000,1000.000000,1000.000000,1000.000000
2021-01-04,991.654606,991.695237,994.789213,994.863812
2021-01-05,997.192527,997.229117,1001.463303,1001.602596
2021-01-06,1007.538380,1007.688334,1007.866147,1008.019546
2021-01-07,1014.586558,1015.174919,1020.115066,1020.360847


In [16]:
# Load Composition & Weights sheet
ethic_weights = pd.read_excel(
    "Solactive_Ethic_Backtest_20260130.xlsx",
    sheet_name="Composition_Weights"
)

print(f"Composition/Weights shape: {ethic_weights.shape}")
print(f"Columns: {list(ethic_weights.columns)}")
ethic_weights.head()

Composition/Weights shape: (113, 8)
Columns: ['Ticker', 'Name', '4/7/2020', '4/7/2021', '4/7/2022', '4/7/2023', '4/5/2024', '4/7/2025']


,Ticker,Name,4/7/2020,4/7/2021,4/7/2022,4/7/2023,4/5/2024,4/7/2025
0,LLY-NYS,Eli Lilly and Company,0.030015,0.027501,0.040249,0.051708,0.080000,0.080000
1,V-NYS,Visa Inc. Class A,0.062145,0.055158,0.054279,0.058151,0.056626,0.076196
2,MA-NYS,Mastercard Incorporated Class A,0.054300,0.053874,0.051146,0.054455,0.057192,0.062574
3,CRM-NYS,"Salesforce, Inc.",0.029134,0.030011,0.031021,0.031654,0.037479,0.032549
4,IBM-NYS,International Business Machines Corporation,0.022268,0.018290,0.017257,0.018841,0.022459,0.029156


---
## 4. Tail-Risk Story
**Question:** How do honorees behave in the worst market months vs. the benchmark?

We identify the worst benchmark months (bottom 5%), then measure:
1. **Average outperformance** — how much less do honorees fall?
2. **Hit rate** — how often does the average honoree beat the benchmark?
3. **Breadth** — what share of individual honorees beat the benchmark?

All metrics are reported with 95% bootstrap confidence intervals.

#### 4a. Identify honoree tickers and filter the prices DataFrame
Match the tickers from the Ethisphere composition/weights sheet to the tickers in our daily prices data.

In [17]:
# Extract base ticker symbols from ethic_weights (format is "LLY-NYS" -> "LLY")
ethic_weights["Ticker_Clean"] = ethic_weights["Ticker"].str.split("-").str[0]

# Find which honoree tickers exist in our prices DataFrame
honoree_tickers = [t for t in ethic_weights["Ticker_Clean"].unique() if t in prices.columns]
missing_tickers = [t for t in ethic_weights["Ticker_Clean"].unique() if t not in prices.columns]

print(f"Honoree tickers matched in prices data: {len(honoree_tickers)}")
print(f"Missing tickers (not in prices data): {len(missing_tickers)} -> {missing_tickers}")

# Filter prices to only honoree stocks
honoree_prices = prices[honoree_tickers]
print(f"\nHonoree prices shape: {honoree_prices.shape}")
honoree_prices.head()

Honoree tickers matched in prices data: 113
Missing tickers (not in prices data): 0 -> []

Honoree prices shape: (1304, 113)


,LLY,V,MA,CRM,IBM,LIN,PEP,OR,ACN,VZ,...,TKR,ITRI,LEA,MOTV3,NATU3,BIMBOA,GT,PINC,SIG,CRI
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-01,159.96,210.51,346.45,219.86,97.20,245.93,127.11,288.85,240.16,42.80,...,70.78,95.90,141.37,11.86,50.10,159.96,10.91,5.39,25.66,77.88
2021-01-04,156.79,209.57,341.16,217.67,95.70,241.54,123.65,289.69,235.79,42.88,...,69.12,95.65,139.95,11.68,49.62,156.79,10.17,5.49,26.02,76.33
2021-01-05,157.57,206.45,337.21,218.86,97.40,245.33,124.02,286.72,237.13,42.68,...,71.20,96.42,142.38,11.49,49.33,157.57,10.64,5.46,27.87,76.86
2021-01-06,155.68,204.63,337.34,213.56,99.83,253.38,122.51,287.00,239.73,43.20,...,76.18,106.98,147.29,11.10,46.67,155.68,10.78,5.50,31.11,83.24
2021-01-07,157.11,205.77,340.01,215.36,99.60,252.66,122.11,285.88,241.99,43.10,...,77.66,107.27,151.48,11.12,46.27,157.11,10.74,5.49,33.10,82.96


#### 4b. Compute monthly returns for honorees and the benchmark
Resample daily prices to month-end and compute percentage returns. For the benchmark we use the SGMACUT (total return) index from the Solactive backtest data.

In [18]:
# Monthly returns for each honoree stock
# Take last available price each month, then compute pct_change
honoree_monthly = honoree_prices.resample("ME").last().pct_change().dropna()

# Monthly benchmark returns from the SGMACUT Index (total return)
bench_monthly = ethic_perf["SGMACUT Index"].resample("ME").last().pct_change().dropna()

# Align dates: keep only months present in both DataFrames
common_months = honoree_monthly.index.intersection(bench_monthly.index)
honoree_monthly = honoree_monthly.loc[common_months]
bench_monthly = bench_monthly.loc[common_months]

print(f"Honoree monthly returns: {honoree_monthly.shape}")
print(f"Benchmark monthly returns: {bench_monthly.shape}")
print(f"Months covered: {common_months.min().strftime('%Y-%m')} to {common_months.max().strftime('%Y-%m')}")
honoree_monthly.head()

Honoree monthly returns: (59, 113)
Benchmark monthly returns: (59,)
Months covered: 2021-02 to 2025-12


,LLY,V,MA,CRM,IBM,LIN,PEP,OR,ACN,VZ,...,TKR,ITRI,LEA,MOTV3,NATU3,BIMBOA,GT,PINC,SIG,CRI
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-28,-0.010709,0.100764,0.118742,-0.040162,0.011852,-0.004628,-0.053994,0.043765,0.037145,0.009921,...,0.039723,0.362939,0.101627,-0.072165,-0.058949,-0.010709,0.593365,0.017316,0.225536,-0.051996
2021-03-31,-0.088190,-0.003077,0.006223,-0.021365,0.120567,0.151774,0.103676,0.079241,0.101007,0.051572,...,0.035982,-0.243859,0.092929,0.148485,0.042669,-0.088190,0.045211,0.042553,0.164603,0.065412
2021-04-30,-0.021662,0.103092,0.074329,0.087087,0.064634,0.020337,0.019147,0.057816,0.052933,0.004437,...,0.033257,0.014552,0.014254,-0.061566,0.012407,-0.021662,-0.020489,0.022449,0.030614,0.223309
2021-05-31,0.097654,-0.025408,-0.056221,0.033749,0.024410,0.051659,0.026174,0.078000,-0.026943,-0.022553,...,0.058274,0.060151,0.051815,0.144330,0.057837,0.097654,0.152237,-0.043912,0.013874,-0.056629
2021-06-30,0.149115,0.028713,0.012512,0.025931,0.019872,-0.034889,0.008919,0.020356,0.044770,-0.008088,...,-0.088913,0.048558,-0.092367,-0.023751,0.100610,0.149115,-0.135149,-0.029228,0.333684,0.009181


#### 4c. Identify tail months (bottom 5% of benchmark returns)
These are the worst market months — the stress periods we want to study.

In [19]:
TAIL_PERCENTILE = 0.05  # Bottom 5% of benchmark months

# Find the return threshold for the worst 5% of months
tail_threshold = bench_monthly.quantile(TAIL_PERCENTILE)
tail_months = bench_monthly[bench_monthly <= tail_threshold]

print(f"Tail threshold (5th percentile): {tail_threshold:.4f} ({tail_threshold*100:.2f}%)")
print(f"Number of tail months: {len(tail_months)} out of {len(bench_monthly)} total months\n")

# Display the tail months with their benchmark returns
tail_df = pd.DataFrame({
    "Benchmark Return": tail_months.values,
    "Benchmark Return (%)": (tail_months.values * 100).round(2)
}, index=tail_months.index.strftime("%Y-%m"))
tail_df.index.name = "Month"
print("Worst market months (bottom 5%):")
print(tail_df.sort_values("Benchmark Return"))

Tail threshold (5th percentile): -0.0552 (-5.52%)
Number of tail months: 3 out of 59 total months

Worst market months (bottom 5%):
         Benchmark Return  Benchmark Return (%)
Month                                          
2022-09         -0.096183                 -9.62
2022-06         -0.085581                 -8.56
2022-04         -0.079266                 -7.93


#### 4d. Compute honoree excess returns in tail months
For each tail month, calculate how each honoree performed relative to the benchmark. This gives us the raw data for our three metrics.

In [20]:
# Honoree returns in tail months only (rows = tail months, cols = honoree tickers)
honoree_tail = honoree_monthly.loc[tail_months.index]
bench_tail = bench_monthly.loc[tail_months.index]

# Excess return = honoree return - benchmark return
# Positive excess return means the honoree fell LESS than the benchmark
excess_tail = honoree_tail.subtract(bench_tail, axis=0)

# Point estimates for the three key metrics:
# 1. Average outperformance: mean excess return across all honorees and tail months
avg_outperformance = excess_tail.mean().mean()

# 2. Hit rate: fraction of (honoree, tail-month) pairs where honoree beat benchmark
hit_rate = (excess_tail > 0).mean().mean()

# 3. Breadth: for each tail month, what share of honorees beat the benchmark? Then average.
breadth = (excess_tail > 0).mean(axis=1).mean()

print("=== Point Estimates (before bootstrap) ===")
print(f"Avg excess return in tail months:  {avg_outperformance*100:+.2f} pp")
print(f"Hit rate (honoree beats benchmark): {hit_rate*100:.1f}%")
print(f"Breadth (avg share beating bench):  {breadth*100:.1f}%")
print(f"\nExcess returns matrix shape: {excess_tail.shape} (tail months x honorees)")
excess_tail

=== Point Estimates (before bootstrap) ===
Avg excess return in tail months:  +1.39 pp
Hit rate (honoree beats benchmark): 59.3%
Breadth (avg share beating bench):  59.3%

Excess returns matrix shape: (3, 113) (tail months x honorees)


,LLY,V,MA,CRM,IBM,LIN,PEP,OR,ACN,VZ,...,TKR,ITRI,LEA,MOTV3,NATU3,BIMBOA,GT,PINC,SIG,CRI
Date,,,,,,,,,,,,,,,,,,,,,
2022-04-30,0.099370,0.040306,0.097459,-0.092064,0.096093,0.055887,0.105171,0.051998,-0.027407,-0.000713,...,0.028790,-0.013748,-0.023507,-0.009113,-0.202713,0.099370,0.011387,-0.099190,0.047622,-0.004924
2022-06-30,0.120020,0.013602,-0.032847,0.115521,0.102474,-0.025569,0.085984,0.087879,0.015848,0.075084,...,-0.045809,0.043341,-0.016493,0.025431,-0.099627,0.120020,-0.085471,0.143404,-0.017526,0.000323
2022-09-30,0.169633,-0.009793,-0.027224,0.017479,0.021136,0.053242,0.050283,0.058312,-0.011845,0.004145,...,0.033535,-0.018782,-0.040533,0.004906,0.117121,0.169633,-0.184644,0.013233,-0.028958,-0.007370


#### 4e. Bootstrap confidence intervals (10,000 iterations)
Resample honorees with replacement to construct 95% CIs for each metric. In each iteration we draw a bootstrap sample of honorees (columns), then recompute the three metrics across all tail months for that sample.

In [21]:
np.random.seed(42)
N_BOOT = 10_000
n_honorees = excess_tail.shape[1]

# Pre-compute numpy arrays for speed
excess_arr = excess_tail.values          # shape: (n_tail_months, n_honorees)
beat_arr = (excess_tail > 0).values      # boolean: did honoree beat benchmark?

# Storage for bootstrap distributions
boot_avg_outperf = np.zeros(N_BOOT)
boot_hit_rate = np.zeros(N_BOOT)
boot_breadth = np.zeros(N_BOOT)

for i in range(N_BOOT):
    # Sample honoree indices with replacement
    idx = np.random.choice(n_honorees, size=n_honorees, replace=True)
    
    # 1. Average excess return across sampled honorees and all tail months
    boot_avg_outperf[i] = excess_arr[:, idx].mean()
    
    # 2. Hit rate: fraction of (month, honoree) pairs where honoree won
    boot_hit_rate[i] = beat_arr[:, idx].mean()
    
    # 3. Breadth: avg across tail months of share of sampled honorees that beat benchmark
    boot_breadth[i] = beat_arr[:, idx].mean(axis=1).mean()

# 95% confidence intervals (2.5th and 97.5th percentiles)
ci_outperf = np.percentile(boot_avg_outperf, [2.5, 97.5])
ci_hit = np.percentile(boot_hit_rate, [2.5, 97.5])
ci_breadth = np.percentile(boot_breadth, [2.5, 97.5])

print("=== Bootstrap 95% Confidence Intervals (10,000 iterations) ===\n")
print(f"Avg excess return in tail months:   [{ci_outperf[0]*100:+.2f} pp,  {ci_outperf[1]*100:+.2f} pp]")
print(f"Hit rate (honoree beats benchmark):  [{ci_hit[0]*100:.1f}%,  {ci_hit[1]*100:.1f}%]")
print(f"Breadth (share beating benchmark):   [{ci_breadth[0]*100:.1f}%,  {ci_breadth[1]*100:.1f}%]")

=== Bootstrap 95% Confidence Intervals (10,000 iterations) ===

Avg excess return in tail months:   [+0.50 pp,  +2.28 pp]
Hit rate (honoree beats benchmark):  [53.1%,  65.2%]
Breadth (share beating benchmark):   [53.1%,  65.2%]


#### 4f. Visualize bootstrap distributions and tail-month performance
Three plots: (1) histograms of bootstrap distributions with CIs marked, (2) bar chart comparing honoree vs benchmark returns in each tail month.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Bootstrap distribution of average excess return ---
ax = axes[0, 0]
ax.hist(boot_avg_outperf * 100, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(ci_outperf[0] * 100, color="red", linestyle="--", label=f"95% CI: [{ci_outperf[0]*100:+.2f}, {ci_outperf[1]*100:+.2f}]")
ax.axvline(ci_outperf[1] * 100, color="red", linestyle="--")
ax.axvline(avg_outperformance * 100, color="black", linewidth=2, label=f"Point est: {avg_outperformance*100:+.2f} pp")
ax.set_xlabel("Average Excess Return (pp)")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap: Avg Excess Return in Tail Months")
ax.legend(fontsize=9)

# --- Plot 2: Bootstrap distribution of hit rate ---
ax = axes[0, 1]
ax.hist(boot_hit_rate * 100, bins=60, color="darkorange", edgecolor="white", alpha=0.8)
ax.axvline(ci_hit[0] * 100, color="red", linestyle="--", label=f"95% CI: [{ci_hit[0]*100:.1f}%, {ci_hit[1]*100:.1f}%]")
ax.axvline(ci_hit[1] * 100, color="red", linestyle="--")
ax.axvline(hit_rate * 100, color="black", linewidth=2, label=f"Point est: {hit_rate*100:.1f}%")
ax.axvline(50, color="gray", linestyle=":", alpha=0.7, label="50% (coin flip)")
ax.set_xlabel("Hit Rate (%)")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap: Hit Rate in Tail Months")
ax.legend(fontsize=9)

# --- Plot 3: Bootstrap distribution of breadth ---
ax = axes[1, 0]
ax.hist(boot_breadth * 100, bins=60, color="seagreen", edgecolor="white", alpha=0.8)
ax.axvline(ci_breadth[0] * 100, color="red", linestyle="--", label=f"95% CI: [{ci_breadth[0]*100:.1f}%, {ci_breadth[1]*100:.1f}%]")
ax.axvline(ci_breadth[1] * 100, color="red", linestyle="--")
ax.axvline(breadth * 100, color="black", linewidth=2, label=f"Point est: {breadth*100:.1f}%")
ax.axvline(50, color="gray", linestyle=":", alpha=0.7, label="50% (coin flip)")
ax.set_xlabel("Breadth (%)")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap: Breadth in Tail Months")
ax.legend(fontsize=9)

# --- Plot 4: Honoree avg return vs benchmark return in each tail month ---
ax = axes[1, 1]
tail_labels = tail_months.index.strftime("%Y-%m")
honoree_avg_by_month = honoree_tail.mean(axis=1) * 100  # average across honorees
bench_by_month = bench_tail * 100

x = np.arange(len(tail_labels))
width = 0.35
ax.bar(x - width/2, bench_by_month.values, width, label="Benchmark", color="indianred", edgecolor="white")
ax.bar(x + width/2, honoree_avg_by_month.values, width, label="Avg Honoree", color="steelblue", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(tail_labels, rotation=45, ha="right")
ax.set_ylabel("Monthly Return (%)")
ax.set_title("Worst Market Months: Honorees vs Benchmark")
ax.legend()
ax.axhline(0, color="black", linewidth=0.5)

plt.tight_layout()
plt.savefig("tail_risk_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to tail_risk_analysis.png")

#### 4g. Robustness check: bottom 10% of benchmark months
Repeat the full analysis using a less extreme threshold (10th percentile) to increase the number of tail months and test whether the pattern holds with a larger sample.

In [ ]:
TAIL_PERCENTILE_10 = 0.10

# Identify bottom 10% months
tail_threshold_10 = bench_monthly.quantile(TAIL_PERCENTILE_10)
tail_months_10 = bench_monthly[bench_monthly <= tail_threshold_10]

print(f"Tail threshold (10th percentile): {tail_threshold_10*100:.2f}%")
print(f"Number of tail months: {len(tail_months_10)} out of {len(bench_monthly)} total\n")

# Excess returns for this wider set of tail months
honoree_tail_10 = honoree_monthly.loc[tail_months_10.index]
bench_tail_10 = bench_monthly.loc[tail_months_10.index]
excess_tail_10 = honoree_tail_10.subtract(bench_tail_10, axis=0)

# Point estimates
avg_outperf_10 = excess_tail_10.mean().mean()
hit_rate_10 = (excess_tail_10 > 0).mean().mean()
breadth_10 = (excess_tail_10 > 0).mean(axis=1).mean()

# Bootstrap
np.random.seed(42)
n_hon_10 = excess_tail_10.shape[1]
exc_arr_10 = excess_tail_10.values
beat_arr_10 = (excess_tail_10 > 0).values

boot_avg_10 = np.zeros(N_BOOT)
boot_hit_10 = np.zeros(N_BOOT)
boot_bread_10 = np.zeros(N_BOOT)

for i in range(N_BOOT):
    idx = np.random.choice(n_hon_10, size=n_hon_10, replace=True)
    boot_avg_10[i] = exc_arr_10[:, idx].mean()
    boot_hit_10[i] = beat_arr_10[:, idx].mean()
    boot_bread_10[i] = beat_arr_10[:, idx].mean(axis=1).mean()

ci_avg_10 = np.percentile(boot_avg_10, [2.5, 97.5])
ci_hit_10 = np.percentile(boot_hit_10, [2.5, 97.5])
ci_bread_10 = np.percentile(boot_bread_10, [2.5, 97.5])

# Print comparison
print("=" * 70)
print(f"{'Metric':<35} {'Bottom 5% (n='+str(len(tail_months))+')':>16} {'Bottom 10% (n='+str(len(tail_months_10))+')':>16}")
print("=" * 70)
print(f"{'Avg excess return (pp)':<35} {avg_outperformance*100:>+14.2f}   {avg_outperf_10*100:>+14.2f}")
print(f"{'  95% CI':<35} [{ci_outperf[0]*100:+.2f}, {ci_outperf[1]*100:+.2f}]   [{ci_avg_10[0]*100:+.2f}, {ci_avg_10[1]*100:+.2f}]")
print(f"{'Hit rate (%)':<35} {hit_rate*100:>14.1f}   {hit_rate_10*100:>14.1f}")
print(f"{'  95% CI':<35} [{ci_hit[0]*100:.1f}, {ci_hit[1]*100:.1f}]     [{ci_hit_10[0]*100:.1f}, {ci_hit_10[1]*100:.1f}]")
print(f"{'Breadth (%)':<35} {breadth*100:>14.1f}   {breadth_10*100:>14.1f}")
print(f"{'  95% CI':<35} [{ci_breadth[0]*100:.1f}, {ci_breadth[1]*100:.1f}]     [{ci_bread_10[0]*100:.1f}, {ci_bread_10[1]*100:.1f}]")
print("=" * 70)

# Per-month breakdown for the 10th percentile
print(f"\nBottom 10% tail months breakdown:")
print(f"{'Month':<10} {'Bench %':>10} {'Avg Hon %':>12} {'Excess pp':>12} {'% Beating':>12}")
print("-" * 58)
for dt in tail_months_10.sort_values().index:
    b = bench_tail_10.loc[dt] * 100
    h = honoree_tail_10.loc[dt].mean() * 100
    e = h - b
    pct = (excess_tail_10.loc[dt] > 0).mean() * 100
    print(f"{dt.strftime('%Y-%m'):<10} {b:>10.2f} {h:>12.2f} {e:>+12.2f} {pct:>11.1f}%")